In [ ]:
%pip install koreanize-matplotlib

In [ ]:
import pandas as pd
import numpy as np

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

In [ ]:
import sys

print("현재 Python:")
print(sys.executable)

!{sys.executable} -m pip install openpyxl

In [ ]:
import pandas as pd

file_path = "/content/drive/MyDrive/research-data/Clinical/extension of Z-Alizadeh sani dataset.xlsx"

df = pd.read_excel(file_path)

print("데이터 크기:", df.shape)

In [ ]:
df.columns.tolist()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
# 문자열 컬럼 실제 값 확인
str_cols = df.select_dtypes(include='str').columns

for col in str_cols:
    print(f"\n[{col}]")
    print(df[col].value_counts(dropna=False))

In [ ]:
int_cols = df.select_dtypes(include='int64').columns

for col in int_cols:
    print(f"\n{'='*40}")
    print(f"[{col}]")
    print(df[col].value_counts().sort_index())

01. 라이브러리
02. 데이터 불러오기
03. 데이터 구조 확인
04. 결측치 확인
05. Target 확인
06. 불필요/Leakage 변수 제거
07. 범주형 변수 Encoding
08. X / y 분리
09. Train / Test Split
10. Logistic Regression 전처리
11. Decision Tree 전처리
12. Logistic Regression Baseline
13. Decision Tree
14. 성능 비교

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [ ]:
print(df['Cath'].value_counts())

In [ ]:
y = df['Cath'].map({
    'Normal': 0,
    'CAD': 1
})

print(y.value_counts())

# binary target
# Normal → 0
# CAD    → 1

In [ ]:
# leakgage / 불필요 변수 제거

drop_cols = [
    'Cath',
    'LAD',
    'LCX',
    'RCA',
    'Exertional CP'
]

X = df.drop(columns=drop_cols)

print("X 크기:", X.shape)

# Cath는 target으로 따로 사용했기 때문에 제거하고, LAD, LCX, RCA는 CAD 예측에 직접적인 결과 정보를 포함하므로 제외

In [ ]:
print(X.columns.tolist())

In [ ]:
# 범주형  처리
categorical_cols = [
    'Sex',
    'BBB',
    'VHD'
]

In [ ]:
# 숫자변환
binary_cols = [
    'Obesity',
    'CRF',
    'CVA',
    'Airway disease',
    'Thyroid Disease',
    'CHF',
    'DLP',
    'Weak Peripheral Pulse',
    'Lung rales',
    'Systolic Murmur',
    'Diastolic Murmur',
    'Dyspnea',
    'Atypical',
    'Nonanginal',
    'LowTH Ang',
    'LVH',
    'Poor R Progression'
]

In [ ]:
X = X.copy()

for col in binary_cols:
    X[col] = X[col].map({
        'N': 0,
        'Y': 1
    })

In [ ]:
X[binary_cols].head()

In [ ]:
X['Sex'] = X['Sex'].map({
    'Male': 1,
    'Fmale': 0
})

In [ ]:
X['Sex'].value_counts()

In [ ]:
print(X.dtypes)

In [ ]:
print("\n문자열 컬럼:")
print(X.select_dtypes(include='str').columns.tolist())

In [ ]:
# train, test 나누기

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
# stratify=y를 사용하면 Train과 Test에 CAD/Normal 비율이 최대한 동일하게 유지

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

# 로지스틱 회귀 학습

In [ ]:
numeric_cols = X_train.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

categorical_cols = ['BBB', 'VHD']

In [ ]:
lr_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(
            drop='first',
            handle_unknown='ignore'
        ), categorical_cols)
    ]
)

In [ ]:
lr_model = Pipeline(
    steps=[
        ('preprocessor', lr_preprocessor),
        ('model', LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]
)

In [ ]:
lr_model.fit(X_train, y_train)

In [ ]:
y_pred_lr = lr_model.predict(X_test)

In [ ]:
y_prob_lr = lr_model.predict_proba(X_test)[:, 1]

In [ ]:
accuracy_lr = accuracy_score(y_test, y_pred_lr)

print("Logistic Regression Accuracy:", accuracy_lr)

In [ ]:
accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr)
recall_lr = recall_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)
roc_auc_lr = roc_auc_score(y_test, y_prob_lr)

print("=== Logistic Regression Baseline ===")
print(f"Accuracy    : {accuracy_lr:.4f}")
print(f"Precision   : {precision_lr:.4f}")
print(f"Recall      : {recall_lr:.4f}")
print(f"F1-score    : {f1_lr:.4f}")
print(f"ROC-AUC     : {roc_auc_lr:.4f}")

In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr)

print(cm_lr)

In [ ]:
tn, fp, fn, tp = cm_lr.ravel()

specificity_lr = tn / (tn + fp)

print(f"Specificity : {specificity_lr:.4f}")

In [ ]:
print("=== Logistic Regression Baseline ===")
print(f"Accuracy    : {accuracy_lr:.4f}")
print(f"Precision   : {precision_lr:.4f}")
print(f"Recall      : {recall_lr:.4f}")
print(f"Specificity : {specificity_lr:.4f}")
print(f"F1-score    : {f1_lr:.4f}")
print(f"ROC-AUC     : {roc_auc_lr:.4f}")

print("\n=== Confusion Matrix ===")
print(cm_lr)

In [ ]:
print(classification_report(
    y_test,
    y_pred_lr,
    target_names=['Normal', 'CAD']
))

In [ ]:
lr_results = pd.DataFrame({
    'Model': ['Logistic Regression'],
    'Accuracy': [accuracy_lr],
    'Precision': [precision_lr],
    'Sensitivity': [recall_lr],
    'Specificity': [specificity_lr],
    'F1-score': [f1_lr],
    'ROC-AUC': [roc_auc_lr]
})

lr_results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Logistic Regression Confusion Matrix
cm_lr = np.array([
    [12, 6],
    [5, 38]
])

fig, ax = plt.subplots(figsize=(8, 6))

im = ax.imshow(cm_lr, cmap='Blues')

# 축 이름
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])

ax.set_xticklabels([
    '정상으로 판단',
    'CAD로 판단'
])

ax.set_yticklabels([
    '실제 정상 환자',
    '실제 CAD 환자'
])

# 각 칸에 숫자 + 설명
labels = [
    ['정상 ▶ 정상\n12명',
     '정상 ▶ CAD\n6명'],
    ['CAD ▶ 정상\n5명',
     'CAD ▶ CAD\n38명']
]

for i in range(2):
    for j in range(2):
        ax.text(
            j, i,
            labels[i][j],
            ha='center',
            va='center',
            fontsize=14
        )

ax.set_xlabel('모델의 판단', fontsize=13)
ax.set_ylabel('실제 환자 상태', fontsize=13)

ax.set_title(
    'Logistic Regression 혼동행렬',
    fontsize=16,
    pad=15
)

fig.colorbar(im, ax=ax, label='환자 수')

plt.tight_layout()
plt.show()

In [ ]:
# Decision Tree
dt_preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_cols),
        ('cat', OneHotEncoder(
            handle_unknown='ignore'
        ), categorical_cols)
    ]
)

dt_preprocessor

In [ ]:
dt_model = Pipeline(
    steps=[
        ('preprocessor', dt_preprocessor),
        ('model', DecisionTreeClassifier(
            random_state=42
        ))
    ]
)

dt_model

In [ ]:
# train 데이터로 학습 실시
dt_model.fit(X_train, y_train)

In [ ]:
y_pred_dt_train = dt_model.predict(X_train)
y_pred_dt_test = dt_model.predict(X_test)

y_prob_dt_train = dt_model.predict_proba(X_train)[:, 1]
y_prob_dt_test = dt_model.predict_proba(X_test)[:, 1]

print("Train 예측 개수 : ", len(y_pred_dt_train))
print("Test 예측 개수 : ", len(y_pred_dt_test))

In [ ]:
accuracy_dt_train = accuracy_score(y_train, y_pred_dt_train)
accuracy_dt_test = accuracy_score(y_test, y_pred_dt_test)

print("Decision Tree Baseline")
print(f"Train Accuracy : {accuracy_dt_train:.4f}")
print(f"Test Accuracy : {accuracy_dt_test:.4f}")
print(f"Accuracy Gap : {accuracy_dt_train - accuracy_dt_test:.4f}")

Train Accuracy 1.0000: 학습 데이터를 완전히 분류
Test Accuracy 0.7049: 새로운 데이터에서는 성능이 크게 하락
Gap 0.2951: 약 29.5%p 차이로, 제한 없는 트리가 학습 데이터를 지나치게 세분화한 상태
LR Test Accuracy 0.8197보다도 현재 DT가 낮음

In [ ]:
# DT baseline 성능 기록

accuracy_dt = accuracy_score(y_test, y_pred_dt_test)
precision_dt = precision_score(y_test, y_pred_dt_test)
recall_dt = recall_score(y_test, y_pred_dt_test)
f1_dt = f1_score(y_test, y_pred_dt_test)
roc_auc_dt = roc_auc_score(y_test, y_prob_dt_test)

cm_dt = confusion_matrix(y_test, y_pred_dt_test)
tn_dt, fp_dt, fn_dt, tp_dt = cm_dt.ravel()

specificity_dt = tn_dt / (tn_dt + fp_dt)

print("Decision Tree Baseline")
print(f"Accuracy : {accuracy_dt:.4f}")
print(f"Precision : {precision_dt:.4f}")
print(f"Recall : {recall_dt:.4f}")
print(f"Specificity : {specificity_dt:.4f}")
print(f"ROC-AUC : {roc_auc_dt:.4f}")

print("\n confusion matrix")
print(cm_dt)

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
import numpy as np

fig, ax = plt.subplots(figsize=(8, 6))

im = ax.imshow(cm_dt, cmap='Oranges')

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])

ax.set_xticklabels([
    '정상으로 판단',
    'CAD로 판단'
])

ax.set_yticklabels([
    '실제 정상 환자',
    '실제 CAD 환자'
])

labels = [
    [f'정상 → 정상\n{tn_dt}명',
     f'정상 → CAD\n{fp_dt}명'],
    [f'CAD → 정상\n{fn_dt}명',
     f'CAD → CAD\n{tp_dt}명']
]

threshold = cm_dt.max() / 2

for i in range(2):
    for j in range(2):
        ax.text(
            j,
            i,
            labels[i][j],
            ha='center',
            va='center',
            fontsize=14,
            color='white' if cm_dt[i, j] > threshold else 'black'
        )

ax.set_xlabel('모델의 판단', fontsize=13)
ax.set_ylabel('실제 환자 상태', fontsize=13)
ax.set_title(
    'Decision Tree Baseline 혼동행렬',
    fontsize=16,
    pad=15
)

fig.colorbar(im, ax=ax, label='환자 수')

plt.tight_layout()
plt.show()

In [ ]:
# 탐색방법 및 후보값 정의

from sklearn.model_selection import GridSearchCV, StratifiedKFold

cv_dt = StratifiedKFold(
    n_splits = 5,
    shuffle=True,
    random_state = 42
)

param_grid_dt = {
    'model__criterion' : ['gini', 'entropy'],
    'model__max_depth' : [2, 3, 4, 5, 6],
    'model__min_samples_split' : [2, 5, 10],
    'model__min_samples_leaf' : [1, 2, 4, 8],
    'model__class_weight' : [None, 'balanced']
}

print("후보 조합 수 : ", 2*5*3*4*2)
print("교차검증을 포함한 총 학습 횟수", 2*5*3*4*2*5)


In [ ]:
grid_search_dt = GridSearchCV(
    estimator = dt_model,
    param_grid = param_grid_dt,
    scoring = 'roc_auc',
    cv = cv_dt,
    n_jobs = -1,
    return_train_score= True,
    refit = True,
    verbose = 1
)

grid_search_dt

In [ ]:
grid_search_dt.fit(X_train, y_train)

In [ ]:
best_index_dt = grid_search_dt.best_index_

best_cv_auc_dt = grid_search_dt.best_score_
best_train_auc_dt = grid_search_dt.cv_results_[
    'mean_train_score'
][best_index_dt]

print("=== Decision Tree Grid Search 결과 ===")
print("최적 하이퍼파라미터:")
print(grid_search_dt.best_params_)

print(f"\n평균 Train ROC-AUC : {best_train_auc_dt:.4f}")
print(f"평균 CV ROC-AUC    : {best_cv_auc_dt:.4f}")
print(
    f"Train-CV Gap       : "
    f"{best_train_auc_dt - best_cv_auc_dt:.4f}"
)

In [ ]:
# 최적 모델을 꺼내 동일한 train/test 데이터 예측값 생성

best_dt_model = grid_search_dt.best_estimator_

y_pred_dt_tuned_train = best_dt_model.predict(X_train)
y_pred_dt_tuned_test = best_dt_model.predict(X_test)
y_prob_dt_tuned_train = best_dt_model.predict_proba(
    X_train
)[:, 1]

y_prob_dt_tuned_test = best_dt_model.predict_proba(
    X_test
)[:, 1]

print("Train 예측 개수 : ", len(y_pred_dt_tuned_train))
print("Test 예측 개수 : ", len(y_pred_dt_tuned_test))

In [ ]:
# 튜닝 DT가 baseline보다 과적합이 줄었는지 확인

accuracy_dt_tuned_train = accuracy_score(
    y_train,
    y_pred_dt_tuned_train
)

accuracy_dt_tuned_test = accuracy_score(
    y_test,
    y_pred_dt_tuned_test
)

accuracy_gap_dt_tuned = (
    accuracy_dt_tuned_train - accuracy_dt_tuned_test
)

print("=== Tuned Decision Tree 과적합 확인 ===")
print(f"Train Accuracy : {accuracy_dt_tuned_train:.4f}")
print(f"Test Accuracy  : {accuracy_dt_tuned_test:.4f}")
print(f"Accuracy Gap   : {accuracy_gap_dt_tuned:.4f}")

In [ ]:
accuracy_dt_tuned = accuracy_score(
    y_test,
    y_pred_dt_tuned_test
)

precision_dt_tuned = precision_score(
    y_test,
    y_pred_dt_tuned_test
)

recall_dt_tuned = recall_score(
    y_test,
    y_pred_dt_tuned_test
)

f1_dt_tuned = f1_score(
    y_test,
    y_pred_dt_tuned_test
)

roc_auc_dt_tuned = roc_auc_score(
    y_test,
    y_prob_dt_tuned_test
)

cm_dt_tuned = confusion_matrix(
    y_test,
    y_pred_dt_tuned_test
)

tn_dt_tuned, fp_dt_tuned, fn_dt_tuned, tp_dt_tuned = (
    cm_dt_tuned.ravel()
)

specificity_dt_tuned = (
    tn_dt_tuned / (tn_dt_tuned + fp_dt_tuned)
)

print("=== Tuned Decision Tree ===")
print(f"Accuracy    : {accuracy_dt_tuned:.4f}")
print(f"Precision   : {precision_dt_tuned:.4f}")
print(f"Recall      : {recall_dt_tuned:.4f}")
print(f"Specificity : {specificity_dt_tuned:.4f}")
print(f"F1-score    : {f1_dt_tuned:.4f}")
print(f"ROC-AUC     : {roc_auc_dt_tuned:.4f}")

print("\n=== Confusion Matrix ===")
print(cm_dt_tuned)

In [ ]:
model_comparison = pd.DataFrame({
    'Logistic Regression': {
        'Accuracy': accuracy_lr,
        'Precision': precision_lr,
        'Recall': recall_lr,
        'Specificity': specificity_lr,
        'F1-score': f1_lr,
        'ROC-AUC': roc_auc_lr
    },
    'DT Baseline': {
        'Accuracy': accuracy_dt,
        'Precision': precision_dt,
        'Recall': recall_dt,
        'Specificity': specificity_dt,
        'F1-score': f1_dt,
        'ROC-AUC': roc_auc_dt
    },
    'DT Tuned': {
        'Accuracy': accuracy_dt_tuned,
        'Precision': precision_dt_tuned,
        'Recall': recall_dt_tuned,
        'Specificity': specificity_dt_tuned,
        'F1-score': f1_dt_tuned,
        'ROC-AUC': roc_auc_dt_tuned
    }
}).T

model_comparison.round(4)

In [ ]:
ax = model_comparison.plot(
    kind='bar',
    figsize=(13, 7),
    ylim=(0, 1.05),
    rot=0,
    colormap='Set2'
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.3f',
        fontsize=8,
        padding=2
    )

ax.set_title(
    'Logistic Regression과 Decision Tree 성능 비교',
    fontsize=16,
    pad=15
)

ax.set_xlabel('모델', fontsize=12)
ax.set_ylabel('성능 점수', fontsize=12)

ax.legend(
    title='평가지표',
    bbox_to_anchor=(1.02, 1),
    loc='upper left'
)

ax.grid(
    axis='y',
    linestyle='--',
    alpha=0.4
)

plt.tight_layout()
plt.show()

In [ ]:
feature_names_dt = (
    best_dt_model
    .named_steps['preprocessor']
    .get_feature_names_out()
)

feature_importance_dt = (
    best_dt_model
    .named_steps['model']
    .feature_importances_
)

importance_df_dt = pd.DataFrame({
    'Feature': feature_names_dt,
    'Importance': feature_importance_dt
})

importance_df_dt['Feature'] = (
    importance_df_dt['Feature']
    .str.replace(r'^(num|cat)__', '', regex=True)
)

importance_df_dt = (
    importance_df_dt
    .sort_values(
        by='Importance',
        ascending=False
    )
    .reset_index(drop=True)
)

importance_df_dt.head(15)

In [ ]:
importance_nonzero_dt = (
    importance_df_dt[
        importance_df_dt['Importance'] > 0
    ]
    .sort_values(
        by='Importance',
        ascending=True
    )
)

fig, ax = plt.subplots(figsize=(9, 6))

bars = ax.barh(
    importance_nonzero_dt['Feature'],
    importance_nonzero_dt['Importance'],
    color='teal',
    alpha=0.8
)

ax.bar_label(
    bars,
    fmt='%.3f',
    padding=4,
    fontsize=10
)

ax.set_title(
    'Tuned Decision Tree 변수 중요도',
    fontsize=16,
    pad=15
)

ax.set_xlabel('변수 중요도', fontsize=12)
ax.set_ylabel('변수', fontsize=12)

ax.set_xlim(
    0,
    importance_nonzero_dt['Importance'].max() * 1.15
)

ax.grid(
    axis='x',
    linestyle='--',
    alpha=0.4
)

plt.tight_layout()
plt.show()